In [1]:
from transformers import BertTokenizer , BertModel
import torch 

# Loading the bert tokenizer and model 

In [3]:
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertModel.from_pretrained('bert-base-uncased')

print("Model loaded")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

Loading weights: 100%|█████████████████████████████████████████████████████| 199/199 [00:00<00:00, 3712.07it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded
Parameters: 109,482,240


# Tokenize a sentence 

In [4]:
sentence = "The animal didn't cross the street because it was too tired."

# Tokenizer converts text to token IDs automatically
inputs = tokenizer(sentence, return_tensors='pt')

In [5]:
print("Tokens:", tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))
print("Input IDs shape:", inputs['input_ids'].shape)

Tokens: ['[CLS]', 'the', 'animal', 'didn', "'", 't', 'cross', 'the', 'street', 'because', 'it', 'was', 'too', 'tired', '.', '[SEP]']
Input IDs shape: torch.Size([1, 16])


#  BERT embeddings

In [7]:
# torch.no_grad() — we're not training, just running inference
with torch.no_grad():
    outputs = model(**inputs)

In [8]:
last_hidden_state = outputs.last_hidden_state
cls_embedding = outputs.pooler_output

In [9]:
print(f"All token embeddings shape: {last_hidden_state.shape}")
print(f"CLS embedding shape: {cls_embedding.shape}")
print(f"\nEmbedding for 'animal' (token index 2):")
print(last_hidden_state[0][2][:8])  # first 8 values of 'animal' embedding


All token embeddings shape: torch.Size([1, 16, 768])
CLS embedding shape: torch.Size([1, 768])

Embedding for 'animal' (token index 2):
tensor([ 0.1893,  0.1212, -0.6873,  0.0295,  0.4091,  0.4435, -0.0315,  0.3195])


# TEsting if context matter or not 

In [12]:
sentence1 = "I deposited money in the bank."      # financial bank
sentence2 = "I sat by the river bank."            # river bank

In [13]:
inputs1 = tokenizer(sentence1, return_tensors='pt')
inputs2 = tokenizer(sentence2, return_tensors='pt')

In [14]:
# Get embeddings
with torch.no_grad():
    outputs1 = model(**inputs1)
    outputs2 = model(**inputs2)

In [15]:
# Find 'bank' token index in each sentence
tokens1 = tokenizer.convert_ids_to_tokens(inputs1['input_ids'][0])
tokens2 = tokenizer.convert_ids_to_tokens(inputs2['input_ids'][0])

In [16]:
print("Sentence 1 tokens:", tokens1)
print("Sentence 2 tokens:", tokens2)

Sentence 1 tokens: ['[CLS]', 'i', 'deposited', 'money', 'in', 'the', 'bank', '.', '[SEP]']
Sentence 2 tokens: ['[CLS]', 'i', 'sat', 'by', 'the', 'river', 'bank', '.', '[SEP]']


In [17]:
bank_idx1 = tokens1.index('bank')
bank_idx2 = tokens2.index('bank')

In [18]:
# Get bank embeddings
bank_emb1 = outputs1.last_hidden_state[0][bank_idx1]
bank_emb2 = outputs2.last_hidden_state[0][bank_idx2]


In [19]:
# Cosine similarity — are they the same?
similarity = torch.nn.functional.cosine_similarity(
    bank_emb1.unsqueeze(0), 
    bank_emb2.unsqueeze(0)
)
print(f"\nSimilarity of 'bank' in two contexts: {similarity.item():.4f}")
print("(1.0 = identical, 0.0 = completely different)")


Similarity of 'bank' in two contexts: 0.5257
(1.0 = identical, 0.0 = completely different)
